# Datasets for the perturbation modeling chapter

The perturbation modeling chapter uses two CRISPR screens that pertpy provides.
The ECCITE-seq screen of Papalexi et al. measures RNA, surface proteins and guide RNAs of THP-1 cells, and the genome-scale Perturb-seq screen of Replogle et al. knocks down essential genes in K562 cells.
To keep the chapter light, we keep all control cells of the Replogle screen but at most 50 cells per perturbation.

In [1]:
import lamindb as ln

ln.connect("theislab/sc-best-practices")

ln.track("4eyFp1vcYWl4")

→ connected lamindb: theislab/sc-best-practices


! It looks like you are running jupyter lab but don't have jupyter-server module installed. Please install it via pip install jupyter-server


→ found notebook perturbation_modeling_dataset.ipynb, making new version


→ created Transform('4eyFp1vcYWl40001', key='perturbation_modeling_dataset.ipynb'), started new Run('pc1liurmPEmKn8b1') at 2026-09-29 22:50:59 UTC


→ notebook imports: lamindb-core==2.10.0 pertpy==1.3.0 scanpy==1.12.4 scipy==1.18.1


In [2]:
import tempfile

import pertpy as pt
import scanpy as sc
import scipy.sparse as sp

sc.settings.datasetdir = tempfile.mkdtemp()

In [3]:
papalexi = pt.dt.papalexi_2021()
papalexi["gdo"].X = sp.csr_matrix(papalexi["gdo"].X)
papalexi

Output()

/home/lheumos/miniforge3/envs/perturbation-modeling/lib/python3.13/site-packages/mudata/_core/mudata.py:988: UserWarning: var_names are not unique. To make them unique, call `.var_names_make_unique`.
  self._update_attr("var", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 20729 × 18776
  obs:	'MULTI_ID', 'HTO_classification', 'guide_ID', 'gene_target', 'NT', 'perturbation', 'replicate', 'S.Score', 'G2M.Score', 'phase'
  var:	'name'
  4 modalities
    rna:	20729 × 18649
      obs:	'nCount_RNA', 'nFeature_RNA', 'percent.mito'
      var:	'name'
      layers:	None
    adt:	20729 × 4
      obs:	'nCount_ADT', 'nFeature_ADT'
      var:	'name'
      layers:	None
    hto:	20729 × 12
      obs:	'nCount_HTO', 'nFeature_HTO', 'technique'
      var:	'name'
      layers:	None
    gdo:	20729 × 111
      obs:	'nCount_GDO'
      var:	'name'
      layers:	None

In [4]:
replogle = pt.dt.replogle_2022_k562_essential()
cells = (
    replogle.obs.groupby("perturbation", observed=True, group_keys=False)
    .apply(
        lambda group: (
            group
            if group.name == "control"
            else group.sample(min(len(group), 50), random_state=0)
        )
    )
    .index
)
replogle = replogle[cells].copy()
replogle.X = sp.csr_matrix(replogle.X, dtype="float32")
replogle

Output()

AnnData object with n_obs × n_vars = 109565 × 8563
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'
    layers: None (.X)

In [5]:
ln.Artifact.from_mudata(
    papalexi,
    key="conditions/perturbation_modeling/papalexi_2021.h5mu",
    description="Papalexi 2021 ECCITE-seq screen of THP-1 cells with RNA, protein, hashtag and guide counts",
).save()
ln.Artifact.from_anndata(
    replogle,
    key="conditions/perturbation_modeling/replogle_2022_k562_essential.h5ad",
    description="Replogle 2022 K562 essential gene Perturb-seq screen with at most 50 cells per perturbation",
).save()

→ returning artifact with same hash: Artifact(uid='31U0vgfuWtjjFAtb0000', key='conditions/perturbation_modeling/papalexi_2021.h5mu', description='Papalexi 2021 ECCITE-seq screen of THP-1 cells with RNA, protein, hashtag and guide counts', suffix='.h5mu', kind='dataset', otype='MuData', size=591169328, hash='n7f-lu2FmS4PI_b9YtBAWh', n_files=None, n_observations=20729, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=194, schema_id=None, created_by_id=2, created_at=2026-09-29 22:40:12 UTC, is_locked=False, version_tag=None, is_latest=True); to track this artifact as an input, use: ln.Artifact.get()


/home/lheumos/miniforge3/envs/perturbation-modeling/lib/python3.13/site-packages/mudata/_core/mudata.py:988: UserWarning: var_names are not unique. To make them unique, call `.var_names_make_unique`.
  self._update_attr("var", axis=1, join_common=join_common)


→ returning artifact with same hash: Artifact(uid='NUlDGQSN2Wod5tsn0000', key='conditions/perturbation_modeling/replogle_2022_k562_essential.h5ad', description='Replogle 2022 K562 essential gene Perturb-seq screen with at most 50 cells per perturbation', suffix='.h5ad', kind='dataset', otype='AnnData', size=3203152037, hash='1lOmrRZLzhRW7920HkNCwp', n_files=None, n_observations=109565, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=194, schema_id=None, created_by_id=2, created_at=2026-09-29 22:40:21 UTC, is_locked=False, version_tag=None, is_latest=True); to track this artifact as an input, use: ln.Artifact.get()


Artifact(uid='NUlDGQSN2Wod5tsn0000', key='conditions/perturbation_modeling/replogle_2022_k562_essential.h5ad', description='Replogle 2022 K562 essential gene Perturb-seq screen with at most 50 cells per perturbation', suffix='.h5ad', kind='dataset', otype='AnnData', size=3203152037, hash='1lOmrRZLzhRW7920HkNCwp', n_files=None, n_observations=109565, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=194, schema_id=None, created_by_id=2, created_at=2026-09-29 22:40:21 UTC, is_locked=False, version_tag=None, is_latest=True)

In [6]:
ln.finish()